# Find Optimal Thresholds for ML-Rule Detector

## Purpose
Find optimal probability thresholds for ML-Rule swing detection by testing on validation data.

This notebook implements **Section 7 (Подтверждение swing)** of ML V1.5 spec:
- Swing подтверждается только если `p > threshold` и прошло `R` баров
- Threshold подбирается так, чтобы модель подтверждала примерно столько же сигналов, сколько их в валидационных данных

## Approach
- Load ML-Rule models (LightGBM trained on rule-based pseudo-labels)
- Split data chronologically: 80% train, 20% validation (no shuffle, same as walk-forward)
- Evaluate threshold range [0.1, 0.9] on validation set
- Select threshold minimizing |confirmed_rate / true_rate - 1.0|
- Save optimal thresholds to `thresholds_ml_rule.json`

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import pickle
import json

# Setup paths
ml_v1_path = Path(r"C:\Users\vgvoz\OneDrive\Рабочий стол\Ophis\market-structure-ml-toolkit-main\ML v1")
data_path = ml_v1_path / "data"
config_path = ml_v1_path / "configs"

print("[1/3] Loading ML-Rule models (LightGBM trained on rule labels)...")
with open(data_path / "model_rule_high_baseline.pkl", "rb") as f:
    model_high = pickle.load(f)

with open(data_path / "model_rule_low_baseline.pkl", "rb") as f:
    model_low = pickle.load(f)

print("[2/3] Loading feature and label data...")

# Try to load features - check which file exists
features_files = ["ETHUSDT_15m_features_reduced.parquet", "ETHUSDT_15m_features.parquet"]
features_file = None
for f in features_files:
    if (data_path / f).exists():
        features_file = f
        print(f"  Found features file: {f}")
        break

if features_file is None:
    raise FileNotFoundError(f"No features file found in {data_path}")

features_df = pd.read_parquet(data_path / features_file)
labels_df = pd.read_parquet(data_path / "swing_labels.parquet")

print(f"  Features shape: {features_df.shape}")
print(f"  Labels shape: {labels_df.shape}")

# Labels has 'timestamp' column - set it as index
if 'timestamp' in labels_df.columns:
    labels_df = labels_df.set_index('timestamp')
    print("  Set 'timestamp' as index for labels")

# Get timezone safely (not all indices have .tz attribute)
features_tz = getattr(features_df.index, 'tz', None)
labels_tz = getattr(labels_df.index, 'tz', None)

print(f"  Features index type: {type(features_df.index).__name__}, tz={features_tz}")
print(f"  Labels index type:   {type(labels_df.index).__name__}, tz={labels_tz}")
print(f"  Features index (first 3): {features_df.index[:3].tolist()}")
print(f"  Labels index (first 3):   {labels_df.index[:3].tolist()}")

# Align indices - convert both to datetime with no timezone
features_df.index = pd.to_datetime(features_df.index)
labels_df.index = pd.to_datetime(labels_df.index)

# Remove timezone if present
if hasattr(features_df.index, 'tz') and features_df.index.tz:
    features_df.index = features_df.index.tz_localize(None)
if hasattr(labels_df.index, 'tz') and labels_df.index.tz:
    labels_df.index = labels_df.index.tz_localize(None)

print(f"\n  After alignment:")
print(f"  Features: {len(features_df)}, range: {features_df.index.min()} to {features_df.index.max()}")
print(f"  Labels: {len(labels_df)}, range: {labels_df.index.min()} to {labels_df.index.max()}")

# Merge on index
data = pd.merge(features_df, labels_df, left_index=True, right_index=True, how='inner')
print(f"\nMerged data shape: {data.shape}")

if len(data) == 0:
    print("ERROR: Merge resulted in empty dataframe!")
    print("Features columns:", features_df.columns.tolist())
    print("Labels columns:", labels_df.columns.tolist())
    raise ValueError("Merge failed - no common indices")

print(f"Date range: {data.index.min()} to {data.index.max()}")

# Chronological split (80/20) - NO SHUFFLE, same as walk-forward
print("\n[3/3] Splitting data (chronological 80/20)...")
split_idx = int(len(data) * 0.8)
train_df = data.iloc[:split_idx].copy()
val_df = data.iloc[split_idx:].copy()

print(f"  Train: {len(train_df)} samples ({train_df.index.min()} to {train_df.index.max()})")
print(f"  Val:   {len(val_df)} samples ({val_df.index.min()} to {val_df.index.max()})")

# Feature list (8 features, no segment_id)
features = ['ret_1', 'ret_3', 'body', 'upper_wick', 'lower_wick', 
            'dist_to_roll_max_20', 'dist_to_roll_min_20', 'vol_50']

# Check which features exist
existing_features = [f for f in features if f in val_df.columns]
missing_features = [f for f in features if f not in val_df.columns]

print(f"\nFeatures check:")
print(f"  Existing: {existing_features}")
if missing_features:
    print(f"  Missing: {missing_features}")

# Generate predictions on validation set
X_val = val_df[existing_features].fillna(0)
print(f"\nX_val shape: {X_val.shape}")
print(f"X_val empty? {X_val.empty}")

if X_val.empty:
    raise ValueError(f"X_val is empty! val_df={len(val_df)}, columns={val_df.columns.tolist()}")

p_high_val = model_high.predict_proba(X_val)[:, 1]
p_low_val = model_low.predict_proba(X_val)[:, 1]
y_high_val = val_df['y_high_rule'].values
y_low_val = val_df['y_low_rule'].values

print(f"\nValidation predictions:")
print(f"  High swings: {len(p_high_val)} samples, base_rate={y_high_val.mean():.4f}")
print(f"  Low swings:  {len(p_low_val)} samples, base_rate={y_low_val.mean():.4f}")

[1/3] Loading ML-Rule models (LightGBM trained on rule labels)...
[2/3] Loading feature and label data...
  Found features file: ETHUSDT_15m_features_reduced.parquet
  Features shape: (139219, 8)
  Labels shape: (139219, 4)
  Set 'timestamp' as index for labels
  Features index type: DatetimeIndex, tz=UTC
  Labels index type:   DatetimeIndex, tz=UTC
  Features index (first 3): [Timestamp('2021-07-05 12:00:00+0000', tz='UTC'), Timestamp('2021-07-05 12:15:00+0000', tz='UTC'), Timestamp('2021-07-05 12:30:00+0000', tz='UTC')]
  Labels index (first 3):   [Timestamp('2021-07-05 12:00:00+0000', tz='UTC'), Timestamp('2021-07-05 12:15:00+0000', tz='UTC'), Timestamp('2021-07-05 12:30:00+0000', tz='UTC')]

  After alignment:
  Features: 139219, range: 2021-07-05 12:00:00 to 2025-06-28 20:30:00
  Labels: 139219, range: 2021-07-05 12:00:00 to 2025-06-28 20:30:00

Merged data shape: (139219, 11)
Date range: 2021-07-05 12:00:00 to 2025-06-28 20:30:00

[3/3] Splitting data (chronological 80/20)...
  T

c:\Users\vgvoz\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.3.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
def evaluate_threshold(p_pred, y_true, threshold):
    """Evaluate metrics at a given threshold.
    
    Key metric = ratio = confirmed_rate / true_rate
    Target: ratio ≈ 1.0 (model predicts ~same rate as data)
    """
    pred = (p_pred >= threshold).astype(int)
    
    if pred.sum() == 0:
        return {
            "threshold": threshold,
            "confirmed_rate": 0.0,
            "true_rate": y_true.mean(),
            "ratio": 0.0,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
        }
    
    tp = ((pred == 1) & (y_true == 1)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    fn = ((pred == 0) & (y_true == 1)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    confirmed_rate = pred.mean()
    true_rate = y_true.mean()
    ratio = confirmed_rate / true_rate if true_rate > 0 else 0
    
    return {
        "threshold": threshold,
        "confirmed_rate": confirmed_rate,
        "true_rate": true_rate,
        "ratio": ratio,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# Scan threshold range [0.1, 0.9] with step 0.05
print("Evaluating threshold range [0.1, 0.9] on validation set...\n")
thresholds_high = np.linspace(0.1, 0.9, 17)
thresholds_low = np.linspace(0.1, 0.9, 17)

results_high = [evaluate_threshold(p_high_val, y_high_val, th) for th in thresholds_high]
results_low = [evaluate_threshold(p_low_val, y_low_val, th) for th in thresholds_low]

df_high = pd.DataFrame(results_high)
df_low = pd.DataFrame(results_low)

print("="*90)
print("HIGH SWING THRESHOLDS (P(swing_high) >= threshold)")
print("="*90)
print(df_high.to_string(index=False))

print("\n" + "="*90)
print("LOW SWING THRESHOLDS (P(swing_low) >= threshold)")
print("="*90)
print(df_low.to_string(index=False))

# Find optimal: minimize |ratio - 1.0|
best_high = df_high.loc[(df_high['ratio'] - 1.0).abs().idxmin()]
best_low = df_low.loc[(df_low['ratio'] - 1.0).abs().idxmin()]

print("\n" + "="*90)
print("OPTIMAL THRESHOLDS")
print("="*90)
print(f"\nSelection criterion: minimize |ratio - 1.0|")
print(f"  (where ratio = confirmed_rate / true_rate)")
print(f"\nHIGH SWINGS:")
print(f"  threshold: {best_high['threshold']:.4f}")
print(f"  ratio:     {best_high['ratio']:.4f}")
print(f"  precision: {best_high['precision']:.4f}")
print(f"  recall:    {best_high['recall']:.4f}")
print(f"  F1:        {best_high['f1']:.4f}")

print(f"\nLOW SWINGS:")
print(f"  threshold: {best_low['threshold']:.4f}")
print(f"  ratio:     {best_low['ratio']:.4f}")
print(f"  precision: {best_low['precision']:.4f}")
print(f"  recall:    {best_low['recall']:.4f}")
print(f"  F1:        {best_low['f1']:.4f}")

optimal_high_th = float(best_high['threshold'])
optimal_low_th = float(best_low['threshold'])

Evaluating threshold range [0.1, 0.9] on validation set...

HIGH SWING THRESHOLDS (P(swing_high) >= threshold)
 threshold  confirmed_rate  true_rate    ratio  precision   recall       f1
      0.10        0.343916   0.118697 2.897428   0.345134 1.000000 0.513159
      0.15        0.343377   0.118697 2.892890   0.345675 1.000000 0.513757
      0.20        0.342767   0.118697 2.887746   0.346291 1.000000 0.514437
      0.25        0.341474   0.118697 2.876853   0.347602 1.000000 0.515882
      0.30        0.339750   0.118697 2.862330   0.349366 1.000000 0.517822
      0.35        0.338170   0.118697 2.849017   0.350361 0.998185 0.518670
      0.40        0.336087   0.118697 2.831467   0.352105 0.996974 0.520414
      0.45        0.333214   0.118697 2.807262   0.354495 0.995159 0.522769
      0.50        0.329119   0.118697 2.772769   0.357595 0.991528 0.525624
      0.55        0.323553   0.118697 2.725870   0.361749 0.986082 0.529316
      0.60        0.312096   0.118697 2.629349   0.36

In [9]:
# Save optimal thresholds to config file (Section 11: Версионирование)
config_file = config_path / "thresholds_ml_rule.json"

config = {
    "threshold_high": optimal_high_th,
    "threshold_low": optimal_low_th,
    "left": 10,
    "right": 10,
    "selection_rule": "ratio_matching_on_validation_80_20_chronological_split"
}

with open(config_file, "w") as f:
    json.dump(config, f, indent=2)

print(f"✓ Saved optimal thresholds to {config_file}")
print(json.dumps(config, indent=2))

print("\n" + "="*90)
print("NEXT STEPS")
print("="*90)
print("\n1. Run walk_forward_rule_vs_ml_rule_vs_ml_fractal.ipynb")
print("   - Sections 1-5: Main walk-forward analysis (est. 40-60 min)")
print("   - Sections 6-8: Results and analysis")
print("\n2. Compare performance metrics:")
print("   - Total PnL")
print("   - # Trades")
print("   - Profit Factor (PF)")
print("   - Max Drawdown (DD)")
print("   - Signal skips")
print("\n3. Criteria (Section 15: Readiness):")
print("   ✓ No-lookahead: Swing confirmed only at t+R")
print("   ✓ Plug-in: Works with existing strategy (rule, ml_rule, ml_fractal modes)")
print("   ✓ No breakage: Entry/exit/risk rules unchanged")
print("   ✓ WFA: Walk-forward analysis complete")

✓ Saved optimal thresholds to C:\Users\vgvoz\OneDrive\Рабочий стол\Ophis\market-structure-ml-toolkit-main\ML v1\configs\thresholds_ml_rule.json
{
  "threshold_high": 0.85,
  "threshold_low": 0.85,
  "left": 10,
  "right": 10,
  "selection_rule": "ratio_matching_on_validation_80_20_chronological_split"
}

NEXT STEPS

1. Run walk_forward_rule_vs_ml_rule_vs_ml_fractal.ipynb
   - Sections 1-5: Main walk-forward analysis (est. 40-60 min)
   - Sections 6-8: Results and analysis

2. Compare performance metrics:
   - Total PnL
   - # Trades
   - Profit Factor (PF)
   - Max Drawdown (DD)
   - Signal skips

3. Criteria (Section 15: Readiness):
   ✓ No-lookahead: Swing confirmed only at t+R
   ✓ Plug-in: Works with existing strategy (rule, ml_rule, ml_fractal modes)
   ✓ No breakage: Entry/exit/risk rules unchanged
   ✓ WFA: Walk-forward analysis complete
